# Indexar ilustraciones → dataset maestro (GitHub + Drive)
## (paso previo al entrenamiento)

Arma el índice maestro de ilustraciones **desde Colab**, combinando dos fuentes y guardando todo **en tu Drive**:

1. **Parciales ya hechos en GitHub** (`datos/ilustraciones/parciales/*.csv`): traen la temática y las
   **descripciones curadas** por cada quien. Son la fuente autoritativa de lo ya recolectado.
2. **Carpetas de imágenes en tu Drive**: las imágenes reales + algunas carpetas nuevas que aún no
   están en GitHub. Si una carpeta trae su `metadata_ilustraciones.csv` se usa; si no, la **temática
   se saca del nombre de la carpeta** (que además identifica a quién la hizo).

Luego junta todo (**parciales de GitHub primero**) y **deduplica por hash**: así las imágenes ya
registradas conservan su metadata rica de GitHub, y las **nuevas del Drive se agregan** con su temática.
El resultado es `dataset_ilustraciones.csv` (mismo esquema del repo) escrito en tu Drive.

> Replica el flujo de `scripts/procesar_ilustraciones.py` + `scripts/fusionar_ilustraciones.py`, pero
> sobre todas las carpetas a la vez y sumando lo que ya hay en GitHub. Las imágenes siguen en Drive,
> que es de donde lee el entrenamiento (`entrenar_ilustraciones_lora.ipynb`).


## Paso 0 — Montar Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## Paso 1 — Traer los parciales del repo
Clona el repo (solo lectura) para leer los `parcial_*.csv` que el equipo ya subió a GitHub.

In [ ]:
import os
!rm -rf /content/repo
!git clone --depth 1 https://github.com/Angel-Eduardo-Reyes-Leon/cuentos-ilustrados-ia.git /content/repo

DIR_PARCIALES_GH = "/content/repo/datos/ilustraciones/parciales"
print("\nParciales en GitHub:")
for f in sorted(os.listdir(DIR_PARCIALES_GH)):
    print("  ", f)


## Paso 2 — Configuración

Ajusta `CARPETA_RAIZ` (subcarpetas de ilustraciones en tu Drive) y `CARPETA_SALIDA` (dónde guardar el
índice, también en Drive). El `MAPEO_CARPETAS` da la temática a las carpetas que **no** traen su
`metadata_ilustraciones.csv`.

In [ ]:
# === Entradas ===
CARPETA_RAIZ = "/content/drive/MyDrive/IMAGENES_CUENTOS"   # <-- CAMBIA: carpeta con las subcarpetas
NOMBRE_DOC   = "metadata_ilustraciones.csv"                # documento esperado por carpeta (si existe)

# === Salida (Drive) ===
CARPETA_SALIDA = "/content/drive/MyDrive/ilustraciones_dataset"
RUTA_MAESTRO   = os.path.join(CARPETA_SALIDA, "dataset_ilustraciones.csv")
DIR_PARCIALES_NUEVOS = os.path.join(CARPETA_SALIDA, "parciales_nuevos")  # solo carpetas nuevas del Drive
RUTA_DESCARTES = os.path.join(CARPETA_SALIDA, "descartados.csv")
os.makedirs(DIR_PARCIALES_NUEVOS, exist_ok=True)

# === Reglas (idénticas al repo) ===
TEMATICAS_VALIDAS = {
    "espacio", "animales", "piratas", "magia_y_brujas", "monstruos_y_criaturas",
    "princesas_y_castillos", "naturaleza_y_bosques", "mar_y_oceano",
    "robots_y_tecnologia", "dinosaurios_y_prehistoria", "fantasmas_y_misterio",
    "heroes_y_aventuras",
}
ESTILO_ACORDADO    = "ilustracion_plana_color"
LADO_MINIMO        = 64
EXTENSIONES_VALIDAS = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}

CAMPOS = ["id", "archivo", "tematica", "estilo", "ancho", "alto",
          "descripcion", "fuente", "recolector", "hash_imagen"]

# Nombre de carpeta → temática (para las carpetas SIN documento)
MAPEO_CARPETAS = {
    "animales_yaretzi": "animales",
    "animales_yaretzi_mexica": "animales",
    "espacio_alejandro_rodea": "espacio",
    "espacio_cristobal_sanchez": "espacio",
    "dinosaurios_y_prehistoria_munguia_cesar": "dinosaurios y prehistoria",
    "prehistoria_dinosarurios_marco-nieves": "dinosaurios y prehistoria",
    "fantasmas_y_misterio_evelyne_rojas": "fantasmas y misterio",
    "fantasmas_y_misterio_Ingrid_Salceda": "fantasmas y misterio",
    "fantasmas y misterio_ Ingrid Salceda": "fantasmas y misterio",
    "heroes_y_aventuras_reynoso_valeria": "heroes y aventuras",
    "magia_y_brujas_karen_flores": "magia y brujas",
    "mis_ilustraciones_moises_cordero": "magia y brujas",
    "mar_y_oceano_Dario_Fuentes": "mar y oceano",
    "mar_y_oceano_fernanda_hernandez": "mar y oceano",
    "monstruos_y_criaturas_fernanda_garcia": "monstruos y criaturas",
    "naturaleza_y_bosques_ernesto_guevara": "naturaleza y bosques",
    "naturaleza_y_bosques_Raul_Hernandez": "naturaleza y bosques",
    "piratas_gabriela_cervantes": "piratas",
    "piratas_Reyna_Alvarez": "piratas",
    "princesas_y_castillos_dayan_garcia": "princesas y castillos",
    "princesas_y_castillos_grisel": "princesas y castillos",
    "mis_ilustraciones": "princesas y castillos",
    "mis_ilustraciones_grisel": "princesas y castillos",
    "robots_y_tecnologia_axel_mendoza": "robots y tecnologia",
    "robots_y_tecnologia_karla_melgarejo": "robots y tecnologia",
    "mis_ilustraciones_antonio": "animales",
    "mis_ilustraciones_axel_mendoza": "robots y tecnologia",
    "mis_ilustraciones_cristobal_sanc": "espacio",
    "mis_ilustraciones_ernesto_guevara": "naturaleza y bosques",
    "mis_ilustraciones_fernanda_garcia": "monstruos y criaturas",
    "mis_ilustraciones_fernanda_hernandez": "mar y oceano",
    "mis_ilustraciones_fuentes_gonzalez": "mar y oceano",
    "mis_ilustraciones_gabriela_cervantes": "piratas",
    "mis_ilustraciones_Hernandez_Raul": "naturaleza y bosques",
    "mis_ilustraciones_karen_flores": "magia y brujas",
    "mis_ilustraciones_karla_melgarejo": "robots y tecnologia",
    "mis_ilustraciones_munguia_cesar": "dinosaurios y prehistoria",
    "mis_ilustraciones_nieves_bartolo": "dinosaurios y prehistoria",
    "mis_ilustraciones_Reyna_Alvarez": "piratas",
    "mis_ilustraciones_reynoso_valeria": "heroes y aventuras",
    "mis_ilustraciones_rojas_evelyne": "fantasmas y misterio",
    "mis_ilustraciones_yaretzi_mexica": "animales",
    "alejandro_rodea": "espacio",
}

print("Raíz Drive:", CARPETA_RAIZ, "| existe:", os.path.isdir(CARPETA_RAIZ))
print("Salida    :", CARPETA_SALIDA)


## Paso 3 — Funciones de validación (mismas reglas del repo)

In [ ]:
import csv, hashlib
from PIL import Image

def hash_archivo(ruta):
    h = hashlib.sha1()
    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(8192), b""):
            h.update(bloque)
    return h.hexdigest()

def normalizar_tematica(t):
    t = (t or "").strip().lower().replace(" ", "_").replace("-", "_")
    while "__" in t:
        t = t.replace("__", "_")
    return t

def tematica_por_carpeta(nombre):
    for clave, tem in MAPEO_CARPETAS.items():
        if nombre == clave or nombre.startswith(clave):
            return normalizar_tematica(tem)
    return None

def leer_documento(carpeta_abs):
    """Registros del metadata de la carpeta, o None si no hay CSV."""
    candidatos = [f for f in os.listdir(carpeta_abs) if f.lower().endswith(".csv")]
    if not candidatos:
        return None
    candidatos.sort(key=lambda x: (x.lower() != NOMBRE_DOC.lower(),
                                   not x.lower().startswith("metadata"), x))
    with open(os.path.join(carpeta_abs, candidatos[0]), encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))

print("Funciones listas.")


## Paso 4 — Indexar las carpetas del Drive

Por cada subcarpeta: usa su documento si lo trae, si no la temática del nombre. Valida cada imagen,
calcula hash y deduplica dentro de la carpeta. `recolector` = nombre de la carpeta.

In [ ]:
# Detecta la tematica desde el nombre de la carpeta (prefijo de tematica + fallback al MAPEO)
def tematica_de_carpeta(nombre):
    n = normalizar_tematica(nombre)
    for tem in sorted(TEMATICAS_VALIDAS, key=len, reverse=True):
        if n.startswith(tem):
            return tem
    return tematica_por_carpeta(nombre)   # casos especiales (prehistoria_dinosarurios, mis_ilustraciones_*)

def indexar_carpeta(carpeta_abs, nombre_carpeta):
    filas, descartados = [], []
    tem = tematica_de_carpeta(nombre_carpeta)

    # descripciones del parcial (si existe), por nombre de archivo — solo para enriquecer
    desc_doc = {}
    registros = leer_documento(carpeta_abs)
    if registros:
        for m in registros:
            a = (m.get("archivo") or "").strip().lower()
            d = (m.get("descripcion") or "").strip()
            if a and d:
                desc_doc[a] = d

    archivos = [a for a in sorted(os.listdir(carpeta_abs))
                if os.path.splitext(a)[1].lower() in EXTENSIONES_VALIDAS]

    if tem not in TEMATICAS_VALIDAS:
        for a in archivos:
            descartados.append((nombre_carpeta, a, f"tematica no resuelta: {tem}"))
        return filas, descartados

    vistos, i = set(), 0
    for a in archivos:
        ruta = os.path.join(carpeta_abs, a)
        try:
            with Image.open(ruta) as img:
                ancho, alto = img.size; img.verify()
        except Exception:
            descartados.append((nombre_carpeta, a, "corrupta")); continue
        if ancho < LADO_MINIMO or alto < LADO_MINIMO:
            descartados.append((nombre_carpeta, a, f"muy chica: {ancho}x{alto}")); continue
        h = hash_archivo(ruta)
        if h in vistos:
            descartados.append((nombre_carpeta, a, "duplicada en la carpeta")); continue
        vistos.add(h); i += 1
        filas.append({"id": f"{nombre_carpeta}_img_{i:04d}", "archivo": a, "tematica": tem,
                      "estilo": ESTILO_ACORDADO, "ancho": ancho, "alto": alto,
                      "descripcion": desc_doc.get(a.lower(), ""), "fuente": "drive",
                      "recolector": nombre_carpeta, "hash_imagen": h})
    return filas, descartados

filas_drive, descartes = [], []
print("Indexando carpetas del Drive...\n")
for nombre in sorted(os.listdir(CARPETA_RAIZ)):
    carpeta_abs = os.path.join(CARPETA_RAIZ, nombre)
    if not os.path.isdir(carpeta_abs):
        continue
    filas, desc = indexar_carpeta(carpeta_abs, nombre)
    filas_drive.extend(filas); descartes.extend(desc)
    if filas:
        with open(os.path.join(DIR_PARCIALES_NUEVOS, f"parcial_{nombre}.csv"),
                  "w", encoding="utf-8", newline="") as f:
            w = csv.DictWriter(f, fieldnames=CAMPOS); w.writeheader(); w.writerows(filas)
    print(f"  {nombre:<45} {len(filas):>5} validas | {len(desc):>4} descart. -> {tematica_de_carpeta(nombre)}")

print(f"\nIndexadas del Drive: {len(filas_drive)} | descartadas: {len(descartes)}")

## Paso 5 — Cargar los parciales de GitHub
Estas filas van **primero** en la fusión, así su descripción curada gana ante una imagen repetida.

In [ ]:
import glob

filas_github = []
for ruta in sorted(glob.glob(os.path.join(DIR_PARCIALES_GH, "*.csv"))):
    with open(ruta, encoding="utf-8-sig", newline="") as f:
        for r in csv.DictReader(f):
            if not (r.get("hash_imagen") or "").strip():
                continue  # sin hash no se puede deduplicar
            filas_github.append({k: (r.get(k) or "") for k in CAMPOS})

print("Filas cargadas de GitHub:", len(filas_github))
print("Recolectores en GitHub:", sorted({f["recolector"] for f in filas_github}))


## Paso 6 — Fusionar (GitHub primero) + dedup por hash → dataset maestro

Mismo criterio que `fusionar_ilustraciones.py`: una imagen por hash. Como GitHub va primero, su fila
gana; las imágenes nuevas del Drive se agregan.

In [ ]:
# Drive es la unica fuente: todas las imagenes estan en subcarpetas con ruta valida.
# (GitHub ya no se usa; si corriste el Paso 5, sus descripciones se aprovechan por hash.)
filas_github = globals().get("filas_github", [])
desc_github = {}
for fl in filas_github:
    h = (fl.get("hash_imagen") or "").strip()
    d = (fl.get("descripcion") or "").strip()
    if h and d:
        desc_github.setdefault(h, d)

por_hash, enriquecidas = {}, 0
for fila in filas_drive:
    h = fila["hash_imagen"]
    if h in por_hash:
        continue
    if not (fila.get("descripcion") or "").strip() and h in desc_github:
        fila["descripcion"] = desc_github[h]; enriquecidas += 1
    por_hash[h] = fila

unicos = list(por_hash.values())

os.makedirs(CARPETA_SALIDA, exist_ok=True)
with open(RUTA_MAESTRO, "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=CAMPOS, extrasaction="ignore")
    w.writeheader(); w.writerows(unicos)
with open(RUTA_DESCARTES, "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f); w.writerow(["carpeta", "archivo", "motivo"]); w.writerows(descartes)

print(f"Dataset maestro (solo Drive, rutas validas): {len(unicos)} ilustraciones -> {RUTA_MAESTRO}")
print(f"  descripciones enriquecidas desde GitHub: {enriquecidas}")
print(f"  descartadas en indexado: {len(descartes)}")

## Paso 7 — Resumen por temática

In [ ]:
from collections import Counter

conteo = Counter(f["tematica"] for f in unicos)
print("Ilustraciones únicas por temática:\n")
for tem, n in sorted(conteo.items(), key=lambda x: -x[1]):
    print(f"  {tem:<28} {n:>6}")
print(f"\n  {'TOTAL':<28} {len(unicos):>6}")

faltantes = TEMATICAS_VALIDAS - set(conteo)
if faltantes:
    print("\nTemáticas sin imágenes:", sorted(faltantes))

if descartes:
    print("\nMotivos de descarte (Drive):")
    for motivo, n in Counter(d[2].split(":")[0] for d in descartes).most_common():
        print(f"  {motivo:<28} {n:>6}")
